# AI Career Copilot — Backend (run this in Google Colab)

This notebook starts the FastAPI backend (`backend.py`) and exposes it publicly with `pyngrok`, so the Streamlit app can call it.

**Steps:**
1. Run all cells in order.
2. When prompted, paste your Cohere API key and your ngrok authtoken (free at https://dashboard.ngrok.com/get-started/your-authtoken).
3. Copy the public URL printed at the end — paste it into the Streamlit app's sidebar as `BACKEND_URL`.

In [ ]:
!pip install -q fastapi uvicorn cohere pyngrok nest-asyncio

In [ ]:
# Upload prompts.py and backend.py from this same folder.
# In Colab: click the folder icon on the left sidebar -> upload prompts.py and backend.py
# (or mount Google Drive and point to where you saved them).

import os
assert os.path.exists('backend.py'), 'Upload backend.py into the Colab file browser first.'
assert os.path.exists('prompts.py'), 'Upload prompts.py into the Colab file browser first.'
print('Found backend.py and prompts.py')

In [ ]:
import os
from getpass import getpass

os.environ['COHERE_API_KEY'] = getpass('Paste your Cohere API key: ')
ngrok_token = getpass('Paste your ngrok authtoken: ')

In [ ]:
import nest_asyncio
from pyngrok import ngrok, conf
import uvicorn

nest_asyncio.apply()
conf.get_default().auth_token = ngrok_token

# Kill any existing tunnels, then open a fresh one on port 8000
ngrok.kill()
public_url = ngrok.connect(8000)
print('Backend public URL:', public_url)
print('Copy this URL into the Streamlit app sidebar (append nothing extra, no trailing slash needed).')

In [ ]:
from backend import app

# Colab already runs an event loop, so uvicorn.run() will raise
# 'asyncio.run() cannot be called from a running event loop'.
# Using uvicorn.Server + await works inside Colab's existing loop instead.
# This cell blocks and keeps the server running. Stop it (interrupt execution)
# to shut the backend down.
config = uvicorn.Config(app, host='0.0.0.0', port=8000, loop='asyncio')
server = uvicorn.Server(config)
await server.serve()